# 0.7 整数类

前面我们已经用列表实现了整数的比较、加法、减法、乘法和除法。这一节把这些能力放进一个 `ListNumber` 类里，让一个列表整数自己保存数据，也自己完成计算。

## 什么是类

类可以理解成一种“自己定义的新类型”。Python 已经有 `int`、`list`、`str` 这些类型。我们也可以自己造一个类型，比如 `ListNumber`。

一个类通常包含两部分：

1. 数据：这个对象保存什么。
2. 方法：这个对象会做什么。

对 `ListNumber([1, 2, 3])` 来说，数据是 `[1, 2, 3]`，表示整数 123。方法可以是比较、加法、减法、乘法和除法。

In [ ]:
class SimpleNumber:
    def __init__(self, digits):
        self.digits = digits

    def show(self):
        return self.digits

x = SimpleNumber([1, 2, 3])
print(x.show())

## 封装整数类的思路

封装就是把相关的数据和操作放在一起。以前我们写 `add_lists(list1, list2)`，需要把两个列表传进去。现在可以写成 `a.add(b)`，意思是：让 `a` 这个列表整数和 `b` 相加。

这个类需要解决几件事：

1. 用 `self.digits` 保存数字列表。
2. 创建对象时去掉开头多余的 0。
3. 用方法实现比较和四则运算。
4. 每次计算都返回新的 `ListNumber`，不修改原来的数。
5. 加上操作符重载，让 `a + b`、`a * b` 这样的写法也能工作。

## 先搭好类的外壳

`__init__` 在创建对象时自动运行。`__repr__` 决定打印对象时看到什么。`__eq__` 支持用 `==` 判断两个 `ListNumber` 是否相等。

In [ ]:
class ListNumber:
    def __init__(self, digits):
        self.digits = self.remove_leading_zeros(digits[:])

    def __repr__(self):
        return f"ListNumber({self.digits})"

    def __eq__(self, other):
        return isinstance(other, ListNumber) and self.digits == other.digits

    @staticmethod
    def remove_leading_zeros(digits):
        while len(digits) > 1 and digits[0] == 0:
            digits.pop(0)
        return digits

print(ListNumber([0, 0, 5]))

## 完整实现

下面的代码把前面几节写过的列表整数算法放进类里。注意：`add`、`subtract`、`multiply`、`divide` 仍然是原来的数学过程，只是它们现在属于 `ListNumber` 这个类。

In [ ]:
class ListNumber:
    """用列表保存一个非负整数。"""

    def __init__(self, digits):
        self.digits = self.remove_leading_zeros(digits[:])

    def __repr__(self):
        return f"ListNumber({self.digits})"

    def __eq__(self, other):
        return isinstance(other, ListNumber) and self.digits == other.digits

    @staticmethod
    def remove_leading_zeros(digits):
        while len(digits) > 1 and digits[0] == 0:
            digits.pop(0)
        return digits

    def compare(self, other):
        left = self.digits[:]
        right = other.digits[:]

        if len(left) > len(right):
            return 1
        if len(left) < len(right):
            return -1

        for i in range(len(left)):
            if left[i] > right[i]:
                return 1
            if left[i] < right[i]:
                return -1

        return 0

    def add(self, other):
        result = []
        carry = 0
        i = len(self.digits) - 1
        j = len(other.digits) - 1

        while i >= 0 or j >= 0 or carry > 0:
            digit_sum = carry
            if i >= 0:
                digit_sum += self.digits[i]
                i -= 1
            if j >= 0:
                digit_sum += other.digits[j]
                j -= 1

            result.insert(0, digit_sum % 10)
            carry = digit_sum // 10

        return ListNumber(result)

    def subtract(self, other):
        if self.compare(other) < 0:
            raise ValueError("本题只处理大数减小数")

        result = []
        borrow = 0
        i = len(self.digits) - 1
        j = len(other.digits) - 1

        while i >= 0:
            top_d = self.digits[i] - borrow
            bottom_d = other.digits[j] if j >= 0 else 0

            if top_d < bottom_d:
                top_d += 10
                borrow = 1
            else:
                borrow = 0

            result.insert(0, top_d - bottom_d)
            i -= 1
            j -= 1

        return ListNumber(result)

    def multiply_by_one_digit(self, one_digit):
        result = []
        carry = 0

        for i in range(len(self.digits) - 1, -1, -1):
            digit_product = self.digits[i] * one_digit + carry
            result.insert(0, digit_product % 10)
            carry = digit_product // 10

        while carry > 0:
            result.insert(0, carry % 10)
            carry = carry // 10

        return ListNumber(result)

    def multiply(self, other):
        result = ListNumber([0])
        zeros_count = 0

        for i in range(len(other.digits) - 1, -1, -1):
            one_digit = other.digits[i]
            partial_result = self.multiply_by_one_digit(one_digit)

            if partial_result.digits != [0]:
                partial_result = ListNumber(partial_result.digits + [0] * zeros_count)

            result = result.add(partial_result)
            zeros_count += 1

        return result

    def divide(self, other):
        if other.digits == [0]:
            raise ValueError("除数不能是 0")

        q_digits = []
        r = ListNumber([0])  # r 是 remainder 的缩写，表示当前余数。

        for d in self.digits:  # d 是 digit 的缩写，表示当前落下来的一位数字。
            r = ListNumber(r.digits + [d])

            q_d = 0  # q_d 是 quotient digit 的缩写，表示当前这一位的商。
            while r.compare(other) >= 0:
                r = r.subtract(other)
                q_d += 1

            q_digits.append(q_d)

        return ListNumber(q_digits), r

现在可以像使用普通对象一样使用 `ListNumber`。每次计算都会得到新的对象。

In [ ]:
a = ListNumber([1, 2, 3])
b = ListNumber([4, 5])

print(a.add(b))
print(a.subtract(ListNumber([7, 8])))
print(a.multiply(b))
print(a.divide(ListNumber([4])))

## 操作符重载

Python 看到 `a + b` 时，会尝试调用 `a.__add__(b)`。这叫操作符重载：让我们自己定义的类也能使用 `+`、`-`、`*`、`//`、`%` 这样的符号。

这里不用重新写算法，只要把操作符转到已经写好的方法上。

In [ ]:
def list_number_lt(self, other):
    return self.compare(other) < 0

def list_number_add(self, other):
    return self.add(other)

def list_number_sub(self, other):
    return self.subtract(other)

def list_number_mul(self, other):
    return self.multiply(other)

def list_number_floordiv(self, other):
    q, r = self.divide(other)  # q 是 quotient 的缩写，r 是 remainder 的缩写。
    return q

def list_number_mod(self, other):
    q, r = self.divide(other)  # q 是 quotient 的缩写，r 是 remainder 的缩写。
    return r

def list_number_divmod(self, other):
    return self.divide(other)

ListNumber.__lt__ = list_number_lt
ListNumber.__add__ = list_number_add
ListNumber.__sub__ = list_number_sub
ListNumber.__mul__ = list_number_mul
ListNumber.__floordiv__ = list_number_floordiv
ListNumber.__mod__ = list_number_mod
ListNumber.__divmod__ = list_number_divmod

In [ ]:
a = ListNumber([1, 2, 3])
b = ListNumber([4, 5])

print(a + b)
print(a - ListNumber([7, 8]))
print(a * b)
print(a // ListNumber([4]))
print(a % ListNumber([4]))
print(divmod(a, ListNumber([4])))

## 对应练习

- [0-7_ex1-NumberClass.py](0-7_ex1-NumberClass.py)：在已有整数类上补全操作符重载。
- [0-7_ex1-NumberClass.solution.py](0-7_ex1-NumberClass.solution.py)：完整参考解答。

## 小结

类把数据和方法放在一起。`ListNumber` 用 `digits` 保存数字，用方法完成计算。操作符重载让这个类用起来更像普通整数：`a.add(b)` 可以写成 `a + b`，`a.divide(b)` 可以拆成 `a // b` 和 `a % b`。